In [1]:
# %reload_ext
%load_ext autoreload
%autoreload 2



from pprint import pprint
from dotenv import load_dotenv
load_dotenv("D:\Repos\Hybrid-PID_RL\.ENV")

import os
os.environ["WANDB_DISABLE_SYMLINKS"] = "true"
import wandb

from Helpers import save_config_to_json
from config import config, config_dict

import numpy as np

import torch
import torch.nn as nn
import torch.functional as F

import gymnasium as gym


# config.DEVICE, torch.__version__, gym.__version__, np.__version__

pprint(config_dict)

{'algo': 'SAC',
 'device': 'cuda',
 'env': {'continuous': True,
         'enable_wind': False,
         'gravity': -10,
         'name': 'LunarLander-v3',
         'turbulence_power': 0,
         'wind_power': 0},
 'n_envs': 1,
 'sac': {'batch_size': 256,
         'buffer_size': 1000000,
         'ent_coef': 'auto',
         'gamma': 0.99,
         'gradient_steps': 1,
         'learning_starts': 10000,
         'lr': 0.0003,
         'policy': 'MlpPolicy',
         'tau': 0.005,
         'train_freq': (1, 'step')},
 'total_timesteps': 500000}


In [ ]:
import wandb
import torch

# Start a temporary online run (so W&B can fetch from the server)
wandb.init(
    project="RL_PID Hybrid Experimentation",
    name="artifact_inspector",
    mode="online"   # <--- required for fetching artifacts
)

run_path = "tanushsrivatsa/RL_PID Hybrid Experimentation/gixozfit"

# Fetch the artifact
artifact = wandb.use_artifact(f"tanushsrivatsa-bits-pilani/RL_PID Hybrid Experimentation/run-aay90xm5-history:v0")#, type="model")
artifact_dir = artifact.download()

# Load the weights (if your model is saved as a SB3 .zip)
import os
for f in os.listdir(artifact_dir):
    print(f)   # See what’s inside the artifact

# If you find a .zip file:
model_path = os.path.join(artifact_dir, "model.zip")

# To load an SB3 model (e.g., SAC)
from stable_baselines3 import SAC
model = SAC.load(model_path)

# Or to inspect raw weights if it’s a .pth
# state_dict = torch.load(f"{artifact_dir}/policy_state_dict.pth", map_location='cpu')

wandb.finish()


CommError: type model specified but this artifact is of type wandb-history

In [3]:
config.wandb.run

'forge-1.3'

In [4]:

# wandb.init(
#     project="RL_PID Hybrid Experimentation",      # name of your project on wandb.ai
#     name="run-1",                   # optional: name of this run
#     config={
#         "learning_rate": 1e-3,
#         "batch_size": 64,
#         "optimizer": "adam",
#         "epochs": 10
#     }
# )


In [5]:

# env = gym.make(config.Environment.name, continuous=config.Environment.continuous, gravity=config.Environment.gravity, 
#                turbulence_power=config.Environment.turbulence_power, enable_wind=config.Environment.enable_wind, 
#                wind_power=config.Environment.wind_power)

### Env Setup

In [6]:
import gymnasium as gym
from stable_baselines3.common.env_util import make_vec_env

#single environment
env = gym.make(config.env.name, continuous=config.env.continuous, gravity=config.env.gravity, 
               turbulence_power=config.env.turbulence_power, enable_wind=config.env.enable_wind, 
               wind_power=config.env.wind_power, render_mode=None)

#allows parallel environments for faster training and SB3 compatibility.
vec_env = make_vec_env(lambda: gym.make(config.env.name, continuous=config.env.continuous, gravity=config.env.gravity, 
               turbulence_power=config.env.turbulence_power, enable_wind=config.env.enable_wind, 
               wind_power=config.env.wind_power, render_mode=None), n_envs=config.n_envs)


c:\Users\91748\.conda\envs\rl_env\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [7]:
obs = vec_env.reset()
print(obs)

[[-4.5242309e-04  1.4085052e+00 -4.5840558e-02 -1.0733305e-01
   5.3103286e-04  1.0383567e-02  0.0000000e+00  0.0000000e+00]]


### SAC From Stable Baselines - 3

In [8]:
import numpy as np
from stable_baselines3.common.callbacks import BaseCallback

from stable_baselines3.common.logger import Logger, KVWriter, make_output_format
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np

class WandbCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.window = 20  # moving average window

    def _on_step(self) -> bool:
        # Track episode rewards
        if len(self.locals.get("infos", [])) > 0:
            for info in self.locals["infos"]:
                if "episode" in info.keys():
                    ep_rew = info["episode"]["r"]
                    self.episode_rewards.append(ep_rew)
                    if len(self.episode_rewards) > self.window:
                        avg_rew = np.mean(self.episode_rewards[-self.window:])
                        wandb.log({"avg_reward": avg_rew}, step=self.num_timesteps)
                    wandb.log({"episode_reward": ep_rew}, step=self.num_timesteps)
        return True

    def _on_training_end(self) -> None:
        # Optional cleanup or summary
        wandb.log({"final_avg_reward": np.mean(self.episode_rewards[-self.window:])})
    
from stable_baselines3.common.logger import configure

def make_wandb_logger():
    tmp_path = "./tmp_logs"
    new_logger = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    return new_logger


In [9]:
import wandb
import numpy as np
from stable_baselines3.common.callbacks import BaseCallback

class WandbMetricsCallback(BaseCallback):
    """
    Logs key metrics from SAC to Weights & Biases:
      - actor loss
      - critic loss
      - entropy loss (temperature)
      - average reward (past n episodes)
      - learning rate
    """
    def __init__(self, window=20, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.window = window

    def _on_step(self) -> bool:
        # Episode reward logging
        infos = self.locals.get("infos", [])
        for info in infos:
            if "episode" in info:
                ep_rew = info["episode"]["r"]
                self.episode_rewards.append(ep_rew)
                avg_rew = np.mean(self.episode_rewards[-self.window:])
                wandb.log(
                    {
                        "episode_reward": ep_rew,
                        "avg_reward": avg_rew,
                    },
                    step=self.num_timesteps,
                )
        return True

    def _on_rollout_end(self):
        """
        SAC stores training info (actor_loss, critic_loss, etc.) in the logger.
        We fetch it here and send to W&B after each rollout.
        """
        try:
            # Extract recent training stats
            log_dict = self.model.logger.name_to_value
            keys = ["train/actor_loss", "train/critic_loss", "train/ent_coef_loss", "train/entropy_loss", "train/learning_rate"]

            metrics = {k.split("/")[-1]: log_dict[k] for k in keys if k in log_dict}
            if metrics:
                wandb.log(metrics, step=self.num_timesteps)
        except Exception as e:
            if self.verbose:
                print(f"[WandbMetricsCallback] Could not extract logger metrics: {e}")

    def _on_training_end(self):
        if len(self.episode_rewards) > 0:
            wandb.log({"final_avg_reward": np.mean(self.episode_rewards[-self.window:])})


In [10]:
from stable_baselines3 import SAC

# Initialize SAC
model = SAC(
    policy=config.sac.policy,          # Standard fully connected neural net
    env=vec_env,                 # The vectorized environment you created
    learning_rate=config.sac.lr,          # Default good LR for SAC
    buffer_size=config.sac.buffer_size,       # Replay buffer size
    learning_starts=config.sac.learning_starts,       # Steps before learning begins
    batch_size=config.sac.batch_size,              # Samples per gradient step
    tau=config.sac.tau,                   # Target smoothing coefficient
    gamma=config.sac.gamma,                  # Discount factor
    train_freq=config.sac.train_freq,      # Learn after every step
    gradient_steps=config.sac.gradient_steps,            # How many gradient updates per step
    ent_coef=config.sac.ent_coef,             # Auto-tunes entropy temperature α
    device=config.device,               # Uses GPU 
    verbose=1                    # Prints training info
)


model.set_logger(make_wandb_logger())

Using cuda device
Logging to ./tmp_logs


### Logging

In [ ]:
import wandb
from wandb.integration.sb3 import WandbCallback
from stable_baselines3.common.callbacks import CheckpointCallback

# --- Initialize W&B run ---
run = wandb.init(
    project="RL_PID Hybrid Experimentation",
    name=config.wandb.run,
    config=config_dict,
    sync_tensorboard=True,   # Sync SB3 logs with wandb
    monitor_gym=True,        # Record environment videos
    save_code=True  ,        # Upload your training script
    # settings=wandb.Settings(_disable_symlinks=True)         
)

# --- Checkpoint saving callback ---
checkpoint_callback = CheckpointCallback(
    save_freq=config.wandb.save_freq,                     
    save_path="./checkpoints/",
    name_prefix="sac_lander",
    verbose=2
)

# --- W&B callback integration ---
wandb_callback = WandbMetricsCallback(40,1)


# wandb_callback = None


# Storing locally for easier navigation
save_config_to_json(config_dict, config.wandb.run)

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: tanushsrivatsa (tanushsrivatsa-bits-pilani) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Config saved to 'configs//forge-1.3.json'


In [ ]:
model.learn(
    total_timesteps=config.total_timesteps,
    log_interval=config.wandb.log_interval,
    callback=wandb_callback,
)

wandb.finish()

wandb: WARNING Step cannot be set when using tensorboard syncing. Please use `run.define_metric(...)` to define a custom metric to log your step values.


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 110      |
|    ep_rew_mean     | -195     |
| time/              |          |
|    episodes        | 10       |
|    fps             | 438      |
|    time_elapsed    | 2        |
|    total_timesteps | 1102     |
| train/             |          |
|    actor_loss      | -1.04    |
|    critic_loss     | 150      |
|    ent_coef        | 0.971    |
|    ent_coef_loss   | -0.0975  |
|    learning_rate   | 0.0003   |
|    n_updates       | 101      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 118      |
|    ep_rew_mean     | -208     |
| time/              |          |
|    episodes        | 20       |
|    fps             | 83       |
|    time_elapsed    | 28       |
|    total_timesteps | 2362     |
| train/             |          |
|    actor_loss      | 3.76     |
|    critic_loss     | 31.8     |
|    ent_coef 

actor_loss,▁▂▂▂▄▃▃▂▃▃▄▄▃▆▃▆▅▄▅▅▆█▅▇▄▆▃▃▆▄▃▅▃▃▅▄▄▆▃▄
avg_reward,▁▆▇▇▆▇▇▇▇▆▆▅▆▅▆▅▆▅▅▅▆▆▆▅▅▅▅▅▅▆▆▇▇███
critic_loss,█▆▄▂▂▂▃▄▅▆▁▂▂▃▂▁▂▂▂▂▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁
ent_coef_loss,███▇▇▆▆▆▆▆▅▅▆▅▃▃▃▂▂▃▂▂▂▂▂▃▁▃▃▃▂▁▃▂▂▃▄▄▄▄
episode_reward,▃▇▆▆▄▆▅▇▃▁▅▄▅▃▇▄▇▃▃▄▇▆▆▄▃▂█▇█▆▇▇█▇▇▇
final_avg_reward,▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
actor_loss,4.23883
avg_reward,-134.34568
critic_loss,2.28357
ent_coef_loss,-0.70769
